In [14]:
import json
import re
import unicodedata
from pathlib import Path

from fuzzywuzzy import process

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_DIR = ROOT / "data"
DB_DIRS = [
    ROOT / "db" / "by_book",
    ROOT / "db" / "by_chapter",
    ROOT / "db" / "by_quran_surah",
]
OUTPUT_DIR = DATA_DIR

SURAH_NAME_PATTERNS = [
    re.compile(r"سورة\s+(.+?)\s+آية\s+(\d+)(?:-(\d+))?", re.I),
    re.compile(r"سورة\s+(.+?)\s+آيات?\s+(\d+)(?:-(\d+))?", re.I),
    re.compile(r"\b(?:Surah|Sura)\s+([A-Za-z0-9'’\-\s]+?)\s+(?:Ayah|ayah|Verse|verse|verses)\s+(\d+)(?:-(\d+))?\b", re.I),
    re.compile(r"\b(?:Surah|Sura)\s+([A-Za-z0-9'’\-\s]+?)\s+(\d+)(?:-(\d+))?\b", re.I),
    re.compile(r"\b(\d{1,3})\s*[:٫]\s*(\d{1,3})\b"),
]

SURAH_FILE_NAME = "{number}-{slug}.json"


def slugify(text: str) -> str:
    text = normalize_text(text)
    return re.sub(r"[^a-z0-9]+", "-", text).strip("-")


def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def normalize_arabic(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]", "", text)
    text = re.sub(r"سورة\s*", "", text)
    text = re.sub(r"آية\s*", "", text)
    text = re.sub(r"[^\u0621-\u064A0-9]+", " ", text)
    return " ".join(text.split())


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def build_surah_lookup():
    arabic_quran = read_json(DATA_DIR / "quran_arabic.json")
    english_quran = read_json(DATA_DIR / "quran_english.json")
    surah_by_number = {}
    surah_by_arabic = {}
    surah_by_english = {}
    english_names = []
    arabic_names = []

    for surah_ar in arabic_quran["data"]["surahs"]:
        number = int(surah_ar["number"])
        surah_en = next((s for s in english_quran["data"]["surahs"] if int(s["number"]) == number), None)
        if surah_en is None:
            continue

        english_name = surah_en.get("englishName", "")
        english_translation = surah_en.get("englishNameTranslation", "")
        arabic_name = surah_ar.get("name", "")

        surah_page = {
            "id": number,
            "name_arabic": arabic_name,
            "name_english": english_name,
            "name_english_translation": english_translation,
            "verses_count": len(surah_ar.get("ayahs", [])),
            "verses_arabic": [ayah.get("text", "") for ayah in surah_ar.get("ayahs", [])],
            "verses_english": [ayah.get("text", "") for ayah in surah_en.get("ayahs", [])],
        }

        surah_by_number[number] = surah_page

        arabic_key = normalize_arabic(arabic_name)
        english_key = normalize_text(english_name)
        translation_key = normalize_text(english_translation)

        if arabic_key:
            surah_by_arabic[arabic_key] = surah_page
            arabic_names.append(arabic_key)
        if english_key:
            surah_by_english[english_key] = surah_page
            english_names.append(english_key)
        if translation_key and translation_key != english_key:
            surah_by_english[translation_key] = surah_page
            english_names.append(translation_key)

    return surah_by_number, surah_by_arabic, surah_by_english, sorted(set(english_names))


def find_surah_by_name(name: str, surah_by_arabic, surah_by_english, english_names):
    if not name:
        return None
    arabic_key = normalize_arabic(name)
    if arabic_key and arabic_key in surah_by_arabic:
        return surah_by_arabic[arabic_key]

    english_key = normalize_text(name)
    if english_key in surah_by_english:
        return surah_by_english[english_key]

    if not english_key:
        return None

    result = process.extractOne(english_key, english_names, score_cutoff=80)
    if result:
        candidate, score = result
        return surah_by_english.get(candidate)

    return None


def parse_reference_text(text: str):
    if not text:
        return []

    matches = []
    for pattern in SURAH_NAME_PATTERNS:
        for match in pattern.finditer(text):
            groups = match.groups()
            if pattern is SURAH_NAME_PATTERNS[-1]:
                surah_number = int(groups[0])
                start_ayah = int(groups[1])
                end_ayah = int(groups[2]) if len(groups) > 2 and groups[2] else start_ayah
                matches.append({
                    "surah_number": surah_number,
                    "start_ayah": start_ayah,
                    "end_ayah": end_ayah,
                    "reference_text": match.group(0),
                    "surah_name": None,
                })
                continue

            surah_name = groups[0].strip()
            start_ayah = int(groups[1])
            end_ayah = int(groups[2]) if len(groups) > 2 and groups[2] else start_ayah
            matches.append({
                "surah_number": None,
                "start_ayah": start_ayah,
                "end_ayah": end_ayah,
                "reference_text": match.group(0),
                "surah_name": surah_name,
            })

    return matches


def parse_hadith_entry(entry, source_name: str):
    arabic_text = entry.get("arabic") if isinstance(entry.get("arabic"), str) else ""
    english_content = entry.get("english")
    english_text = english_content.get("text") if isinstance(english_content, dict) else english_content if isinstance(english_content, str) else ""

    reference_candidates = []
    for text in (english_text, arabic_text):
        reference_candidates.extend(parse_reference_text(text))

    if not reference_candidates:
        return []

    hadith_obj = {
        "source": source_name,
        "id": entry.get("id"),
        "idInBook": entry.get("idInBook"),
        "bookId": entry.get("bookId"),
        "chapterId": entry.get("chapterId"),
        "arabic": arabic_text,
        "english": english_text,
        "narrator": english_content.get("narrator") if isinstance(english_content, dict) else "",
    }

    output = []
    for candidate in reference_candidates:
        output.append({
            "surah_number": candidate["surah_number"],
            "surah_name": candidate["surah_name"],
            "start_ayah": candidate["start_ayah"],
            "end_ayah": candidate["end_ayah"],
            "reference_text": candidate["reference_text"],
            "hadith": hadith_obj,
        })

    return output


def load_hadith_file(path: Path):
    data = read_json(path)
    source_name = path.stem
    if isinstance(data, dict) and "hadiths" in data and isinstance(data["hadiths"], list):
        return source_name, data["hadiths"]
    elif isinstance(data, dict) and "verses" in data and isinstance(data["verses"], list):
        # Extract hadiths from verses in processed format
        hadiths = []
        for verse in data["verses"]:
            if "hadiths" in verse and isinstance(verse["hadiths"], list):
                hadiths.extend(verse["hadiths"])
        return source_name, hadiths
    return None, []


def collect_references():
    surah_by_number, surah_by_arabic, surah_by_english, english_names = build_surah_lookup()

    verse_links = {}
    total_hadiths = 0
    mapped_hadiths = 0

    for db_dir in DB_DIRS:
        for path in sorted(db_dir.rglob("*.json")):
            source_name, entries = load_hadith_file(path)
            if not entries:
                continue

            for entry in entries:
                total_hadiths += 1
                references = parse_hadith_entry(entry, source_name)
                if not references:
                    continue

                mapped_any = False
                for ref in references:
                    if ref["surah_number"] is not None:
                        surah_page = surah_by_number.get(ref["surah_number"])
                    else:
                        surah_page = find_surah_by_name(ref["surah_name"], surah_by_arabic, surah_by_english, english_names)

                    if not surah_page:
                        continue

                    mapped_any = True
                    for ayah_number in range(ref["start_ayah"], ref["end_ayah"] + 1):
                        if ayah_number < 1 or ayah_number > surah_page["verses_count"]:
                            continue
                        key = (surah_page["id"], ayah_number)
                        verse_links.setdefault(key, []).append({
                            "reference_text": ref["reference_text"],
                            "surah_name": ref["surah_name"],
                            "start_ayah": ref["start_ayah"],
                            "end_ayah": ref["end_ayah"],
                            "hadith": ref["hadith"],
                        })
                if mapped_any:
                    mapped_hadiths += 1

    return verse_links, surah_by_number, total_hadiths, mapped_hadiths


def build_output():
    verse_links, surah_by_number, total_hadiths, mapped_hadiths = collect_references()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    summary = {
        "surahs_written": 0,
        "total_hadiths": total_hadiths,
        "mapped_hadiths": mapped_hadiths,
        "verses_with_hadiths": len(verse_links),
    }

    for surah_id, surah in surah_by_number.items():
        verse_data = []
        for ayah_index in range(1, surah["verses_count"] + 1):
            key = (surah_id, ayah_index)
            hadiths = verse_links.get(key, [])
            # Include all verses, even if no hadiths were found
            verse_data.append({
                "verse_number": ayah_index,
                "text_arabic": surah["verses_arabic"][ayah_index - 1],
                "text_english": surah["verses_english"][ayah_index - 1],
                "hadiths": hadiths,
            })

        if not verse_data:
            continue

        file_name = SURAH_FILE_NAME.format(number=surah_id, slug=slugify(surah["name_english"]))
        output_path = OUTPUT_DIR / file_name
        with output_path.open("w", encoding="utf-8") as handle:
            json.dump({
                "surah": {
                    "id": surah_id,
                    "name_arabic": surah["name_arabic"],
                    "name_english": surah["name_english"],
                    "name_english_translation": surah["name_english_translation"],
                    "verses_count": surah["verses_count"],
                },
                "verses": verse_data,
            }, handle, ensure_ascii=False, indent=2)
        summary["surahs_written"] += 1

    print(json.dumps(summary, indent=2, ensure_ascii=False))

 
build_output()


{
  "surahs_written": 114,
  "total_hadiths": 101769,
  "mapped_hadiths": 1434,
  "verses_with_hadiths": 509
}
